In [61]:

from tsl.datasets import Elergone
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure


In [62]:
dataset = Elergone(root='./data')


In [63]:
connectivity = dataset.get_connectivity(threshold=None,
                                        include_self=True,
                                        normalize_axis=1,
                                        layout="dense")


threshold = np.quantile(connectivity.flatten(), 0.8)

adj = connectivity > threshold

np.sum(adj) / adj.size

np.float64(0.2)

In [64]:
print(f"Sampling period: {dataset.freq}")
print(f"Has missing values: {dataset.has_mask}")
print(f"Percentage of missing values: {(1 - dataset.mask.mean()) * 100:.2f}%")
print(f"Has exogenous variables: {dataset.has_covariates}")
print(f"Covariates: {', '.join(dataset.covariates.keys())}")

Sampling period: <15 * Minutes>
Has missing values: True
Percentage of missing values: 20.15%
Has exogenous variables: False
Covariates: 


In [65]:
df = dataset.dataframe()

df.head()

nodes,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
channels,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2011-01-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
print(len(df))
df = df.resample('1h').sum()
print(len(df))
df.head()

140256
35065


nodes,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
channels,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2011-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 04:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
all_val = df.values
T, V = all_val.shape
all_val = all_val.reshape(T, V, 1)

all_val.shape


(35065, 370, 1)

In [68]:

idx_intersection = -1


def has_zero_subarray(arr, min_length, eps=1e-9):
    if len(arr) < min_length:
        return False
    
    # Count consecutive zeros
    max_zeros = 0
    current_zeros = 0
    
    for val in arr:
        if val < eps:
            current_zeros += 1
            max_zeros = max(max_zeros, current_zeros)
        else:
            current_zeros = 0
    
    return max_zeros >= min_length


all_val_nz = all_val[10000:, :, :]
print(all_val_nz.shape)
node_idx_to_keep = []

for node_idx in range(all_val_nz.shape[1]):
    values = all_val_nz[:, node_idx, 0]
    if not has_zero_subarray(values, 75, 1e-2): 
        node_idx_to_keep.append(node_idx)


all_val_nz = all_val_nz[:, node_idx_to_keep, :]

all_val_nz.shape

(25065, 370, 1)


(25065, 316, 1)

In [69]:
adj = adj[np.ix_(node_idx_to_keep, node_idx_to_keep)]

adj.shape

(316, 316)

In [70]:

PLOT = False

rand_idx = np.random.randint(0, all_val_nz.shape[1], 50)


if PLOT:

    for node_idx in range(all_val_nz.shape[1]):
        figure(figsize=(10,  6), dpi=80)
        plt.plot(all_val_nz[:, node_idx, 0])
        plt.title(f"Node {node_idx}")
        plt.show()

In [71]:
DATA_PATH = './diffstg/data/dataset/electricity_benchmark/'

np.save(os.path.join(DATA_PATH, 'flow.npy'), all_val_nz)
np.save(os.path.join(DATA_PATH, 'adj.npy'), adj)

In [72]:
adj = np.load(os.path.join(DATA_PATH, 'adj.npy'))

flow = np.load(os.path.join(DATA_PATH, 'flow.npy'))

adj.shape, flow.shape

((316, 316), (25065, 316, 1))